
# Overview of Linear Regression

**Goals**
- Build intuition for linear regression.
- Fit a line to data using the **closed-form solution** and **gradient descent**.
- Evaluate model fit with MSE and R².
- Compare with `scikit-learn` and peek at **L2 regularization (Ridge)**.

> Run cells top-to-bottom. This notebook uses only standard Python libraries plus `numpy`, `matplotlib`, and `scikit-learn`.


In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)     #creates a repeatable (same seed = 42) random number generator
np.set_printoptions(precision=4, suppress=True)



## 1) Create a simple synthetic dataset

We'll sample inputs `x` uniformly and generate outputs `y = 3.5 x + 2 + error`.


In [ ]:
# Generate synthetic data
n = 100                              # number of samples to simulate
x = rng.uniform(-3, 3, size=n)       # sample X values uniformally from -3 to 3
error = rng.normal(0, 1.0, size=n)   # set the mean, SD, and distribution of error to add to the Y values
y = 3.5 * x + 2.0 + error            # generate Y values

# Save for reuse (optional)
import pandas as pd
df = pd.DataFrame({'x': x, 'y': y})
df.to_csv('linreg_data.csv', index=False)   # export the simulated data to csv file
print('Saved data to linreg_data.csv')


<div align="left">
  <img src="https://media.geeksforgeeks.org/wp-content/uploads/20231129130431/11111111.png" width="600">

  ## **Why is linear regression a considered "supervised learning" approach?**


#### 2) "Closed-" solution for linear regression. (i.e. exact mathematical formula; no iteration)

### Simple Linear Regression (one predictor)
For a single predictor $x$ and response $y$:
$$
\hat{y}_i = b_0 + b_1 x_i
$$

The least-squares solution gives:
$$
b_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2},
\qquad
b_0 = \bar{y} - b_1 \bar{x}
$$
That’s the closed-form solution in **scalar algebra** — one slope and one intercept.

---

### Expressing the same thing in matrix form
We can write all $n$ data points at once:
$$
\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon}
$$

where
$$
\mathbf{X} =
\begin{bmatrix}
1 & x_1 \\
1 & x_2 \\
\vdots & \vdots \\
1 & x_n
\end{bmatrix},\quad
\boldsymbol{\beta} =
\begin{bmatrix}
b_0 \\ b_1
\end{bmatrix},\quad
\mathbf{y} =
\begin{bmatrix}
y_1 \\ y_2 \\ \vdots \\ y_n
\end{bmatrix}
$$
The column of 1s in $\mathbf{X}$ will represent the intercept.

Then minimizing $\,\lVert \mathbf{y} - \mathbf{X}\boldsymbol{\beta}\rVert^2\,$ gives:
$$
\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1}\mathbf{X}^\top \mathbf{y}
$$

## Solving that minimization gives the **closed-form OLS estimator**:

---

### The same thing just different representations
For the $2\times n$ case,
$$
\mathbf{X}^\top \mathbf{X} =
\begin{bmatrix}
n & \sum x_i \\
\sum x_i & \sum x_i^2
\end{bmatrix},
\qquad
\mathbf{X}^\top \mathbf{y} =
\begin{bmatrix}
\sum y_i \\
\sum x_i y_i
\end{bmatrix}
$$

Solving yields:
$$
\hat{\boldsymbol{\beta}} =
\begin{bmatrix}
\bar{y} - b_1 \bar{x} \\
b_1
\end{bmatrix}  \text{ which is}
\quad
\begin{bmatrix}
b_0 \\ b_1
\end{bmatrix} \text{ from the scalar solution above}
$$

— exactly the same slope and intercept as before!

---

### Worth noting...
| Setting | Formula for coefficients | Interpretation |
|---|---|---|
| **One predictor** | $ b_1 = \frac{\mathrm{Cov}(x,y)}{\mathrm{Var}(x)} $ | slope of best-fit line |
| **Many predictors** | $ \hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1}\mathbf{X}^\top \mathbf{y} $ | slopes of best-fit hyperplane |

The matrix equation for OLS is just the compact version of the same idea — **fit the line (or plane) that best predicts $y$ from $X$.**


In [ ]:
#@title Visualize data and OLS minimizing MSE
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Load dataset ---
df = pd.read_csv("linreg_data.csv")
x = df["x"].values
y = df["y"].values

# --- Plot the data ---
plt.figure(figsize=(6,4))
plt.scatter(x, y, alpha=0.7, color="royalblue")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Data loaded from linreg_data.csv")
plt.grid(True)
plt.show()

# --- Define MSE as a function of slope ---
def mse_for_slope(b1):
    """Compute MSE when using slope b1, fitting intercept optimally."""
    b0 = y.mean() - b1 * x.mean()   # best intercept for that slope
    y_pred = b0 + b1 * x
    return np.mean((y - y_pred)**2) # MSE

# --- Sweep through slope values to compute MSE curve ---
slopes = np.linspace(-2, 6, 300)
mses = [mse_for_slope(b) for b in slopes]  #compute MSE for range of b1 values

# --- Compute OLS (closed-form) slope & intercept ---
x_mean, y_mean = x.mean(), y.mean()
b1_hat = np.sum((x - x_mean)*(y - y_mean)) / np.sum((x - x_mean)**2)
b0_hat = y_mean - b1_hat * x_mean
mse_min = mse_for_slope(b1_hat)

# --- Plot the MSE curve with OLS minimum ---
plt.figure(figsize=(6,4))
plt.plot(slopes, mses, color="purple", lw=2)
plt.axvline(b1_hat, color="crimson", linestyle="--", lw=2,
            label=f"OLS slope = {b1_hat:.2f}")
plt.scatter([b1_hat], [mse_min], color="crimson", zorder=5)
plt.xlabel("Slope (b₁)")
plt.ylabel("Mean Squared Error")
plt.title("OLS chooses the slope that minimizes MSE")
plt.legend()
plt.grid(True)
plt.show()

# --- Show the best-fit line on top of the data ---
x_line = np.linspace(x.min(), x.max(), 100)
y_pred = b0_hat + b1_hat * x_line

plt.figure(figsize=(6,4))
plt.scatter(x, y, alpha=0.7, color="royalblue", label="data")
plt.plot(x_line, y_pred, color="crimson", lw=2, label="OLS best-fit line")
plt.xlabel("x")
plt.ylabel("y")
plt.title("OLS regression line (minimizes MSE)")
plt.legend()
plt.grid(True)
plt.show()

# --- Print numeric results ---
print(f"OLS estimates: b0 = {b0_hat:.3f}, b1 = {b1_hat:.3f}")
print(f"Minimum MSE = {mse_min:.3f}")



## 3) Gradient Descent approach to finding $b_1$:

We minimize MSE by iteratively updating parameters.

Updates for learning rate `η`:
$$
\begin{aligned}
b_0 &\leftarrow b_0 - \eta \cdot \frac{2}{n} \sum_i (\hat{y}_i - y_i) \\
b_1 &\leftarrow b_1 - \eta \cdot \frac{2}{n} \sum_i (\hat{y}_i - y_i) x_i
\end{aligned}
$$


In [ ]:
#@title Run Gradient Descent on MSE (slow learning rate for illustration)
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Ensure x, y exist (load if needed)
try:
    x, y
except NameError:
    df = pd.read_csv("linreg_data.csv")
    x = df["x"].values
    y = df["y"].values

# Build design matrix and column vector for y
X = np.c_[np.ones_like(x), x]      # (n, 2) -> [1, x]
y_vec = y.reshape(-1, 1)           # (n, 1)

# OLS (closed-form) for comparison
beta_hat = np.linalg.inv(X.T @ X) @ (X.T @ y_vec)   # (2,1)
b0_ols, b1_ols = beta_hat.ravel()

def gd_linear_regression(X, y, steps=2000, eta=None, record_every=50):
    """
    Minimize MSE(beta) = (1/n)||y - X beta||^2 via Gradient Descent.
    Returns: beta_final, history (list of dicts with step, mse, b0, b1), eta_used
    """
    n = X.shape[0]
    # Auto learning rate (stable) if not provided
    if eta is None:
        XtX = X.T @ X
        L = (2.0 / n) * np.linalg.eigvalsh(XtX).max()
        eta = 0.9 / L  # use 0.1/L if you want slower auto-steps

    beta = np.zeros((X.shape[1], 1))  # start at zeros [b0, b1]
    history = []
    for t in range(steps):
        y_hat = X @ beta
        grad = (-2.0 / n) * (X.T @ (y - y_hat))   # y is (n,1); y_hat is (n,1)
        beta = beta - eta * grad

        if (t % record_every == 0) or (t == steps - 1):
            mse = float(np.mean((y - y_hat) ** 2))
            # Explicit scalar extraction to avoid NumPy deprecation warnings
            history.append({
                "step": t,
                "mse": mse,
                "b0": float(beta[0, 0]),
                "b1": float(beta[1, 0]),
            })
    return beta, history, eta

# Run GD with slower learning rate (eta=?) so you can see the progression
beta_gd, hist, eta_used = gd_linear_regression(X, y_vec, steps=2000, eta=0.0002, record_every=100)
print(f"GD learning rate used: {eta_used:.6g}")
print(f"GD final params -> b0 = {hist[-1]['b0']:.3f}, b1 = {hist[-1]['b1']:.3f}")
print(f"Final MSE (GD): {hist[-1]['mse']:.3f}")

# Plot data, OLS line, and GD snapshots
x_line = np.linspace(x.min(), x.max(), 200)
y_line_ols = b0_ols + b1_ols * x_line

plt.figure(figsize=(7,5))
plt.scatter(x, y, alpha=0.7, label="data")
plt.plot(x_line, y_line_ols, lw=3, label="OLS (closed-form)", color="black", alpha=0.7)

checkpoints = [hist[0], hist[len(hist)//3], hist[2*len(hist)//3], hist[-1]]
for snap in checkpoints:
    y_line_snap = snap["b0"] + snap["b1"] * x_line
    lbl = f"GD step {snap['step']} (b0={snap['b0']:.2f}, b1={snap['b1']:.2f})"
    plt.plot(x_line, y_line_snap, lw=1.8, ls="--", label=lbl)

plt.xlabel("x"); plt.ylabel("y")
plt.title("GD approaches the OLS solution")
plt.legend(fontsize=9)
plt.grid(True)
plt.show()

# MSE vs iteration
steps_ = [h["step"] for h in hist]
mses_  = [h["mse"]  for h in hist]

plt.figure(figsize=(6,4))
plt.plot(steps_, mses_, lw=2)
plt.xlabel("Iteration")
plt.ylabel("MSE")
plt.title("Gradient Descent convergence (MSE ↓)")
plt.grid(True)
plt.show()

print(f"OLS params -> b0 = {b0_ols:.3f}, b1 = {b1_ols:.3f}")
print(f"GD  params -> b0 = {hist[-1]['b0']:.3f}, b1 = {hist[-1]['b1']:.3f}")


##Reasons to choose Gradient Descent over OLS
**Large datasets:** Gradient descent is faster because it doesn't require loading the entire dataset into memory at once, making it suitable for "online" or "streaming" learning.

**High-dimensional data:** When the number of features is very large, the matrix inversion required by OLS can be computationally expensive and slow. Gradient descent can handle this more efficiently.

**Regularization:** Gradient descent is the standard optimization method for models that use regularization (like Lasso or Ridge regression) because it can incorporate the regularization term into its cost function to penalize large coefficients.

**Non-linear problems:** Gradient descent is an iterative method that can be used for more complex models and non-linear functions where an analytical, closed-form solution like OLS's doesn't exist.

**Memory constraints:** It is a good choice when you have limited memory, as you can perform the calculation on smaller batches of data.


## 4) Metrics: MSE and R²

- **MSE**: average squared error. Lower is better.
- **R²**: proportion of variance in `y` explained by the model.


In [ ]:
# Define metrics
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    return 1 - ss_res/ss_tot

# Closed-form predictions & metrics
y_pred_cf = b0_ols + b1_ols * x
print(f"Closed-form -> MSE = {mse(y, y_pred_cf):.3f}, R² = {r2(y, y_pred_cf):.3f}")

# Gradient Descent predictions & metrics
b_gd = hist[-1]['b0']
w_gd = hist[-1]['b1']
y_pred_gd = b_gd + w_gd * x
print(f"Gradient Descent -> MSE = {mse(y, y_pred_gd):.3f}, R² = {r2(y, y_pred_gd):.3f}")



## 5) Comparison with `scikit-learn`

[**scikit-learn**](https://scikit-learn.org/stable/) is one of the most widely used Python libraries for machine learning and statistical modeling.  
It provides efficient, easy-to-use implementations of many algorithms — including linear regression, logistic regression, clustering, dimensionality reduction, and more.

Above we just coded the linear regression from scratch (both closed-form and via gradient descent).

Now we’ll use scikit-learn’s built-in tools to confirm that the library produces identical results — and to see how it extends easily to regularized models like Ridge Regression.

In [ ]:

from sklearn.linear_model import LinearRegression, Ridge

# scikit-learn Linear Regression
lm = LinearRegression()
lm.fit(x.reshape(-1,1), y)
print(f'sklearn LinearRegression -> b = {lm.intercept_:.3f}, w = {lm.coef_[0]:.3f}')

# Metrics
y_pred_skl = lm.predict(x.reshape(-1,1))
print(f'sklearn -> MSE = {mse(y, y_pred_skl):.3f}, R^2 = {r2(y, y_pred_skl):.3f}')



## 6) Brief peek: Ridge Regression — adding regularization to linear models

**Ridge Regression** (also called L2-regularized regression) is a direct extension of Ordinary Least Squares (OLS) that helps when predictors are highly correlated or when we have more features than samples.

---

### The idea

OLS minimizes only the sum of squared errors:
$$
\min_{\boldsymbol{\beta}} \; \|y - X\boldsymbol{\beta}\|^2
$$

Ridge adds a penalty on the size of the coefficients to discourage very large weights:
$$
\min_{\boldsymbol{\beta}} \; \|y - X\boldsymbol{\beta}\|^2 + \alpha \|\boldsymbol{\beta}\|^2
$$

Here:
- \( $\alpha$ \) is a regularization strength (a positive constant).  
- Larger \( $\alpha$ \) means stronger penalty → coefficients are shrunk toward zero.  
- When \( $\alpha$ = 0 \), Ridge reduces to plain OLS.

---

### The closed-form solution

Ridge has a closed-form estimator that looks almost identical to OLS:
$$
\hat{\boldsymbol{\beta}}_{\text{ridge}}
= (X^\top X + \alpha I)^{-1} X^\top y
$$

That extra \( $\alpha I$ \) term:
- Ensures \(X^\top X + $\alpha I$\) is always invertible → improves numerical stability.  
- Shrinks coefficients → lowers variance but introduces a little bias (the bias–variance trade-off).

---

### Intuition

| Concept | OLS | Ridge |
|----------|-----|--------|
| Objective | minimize squared error | minimize squared error + penalty on coefficient size |
| Sensitivity | can overfit/noisy | smoother, more stable |
| Coefficients | can be large/unreliable | smaller, more conservative |
| Hyperparameter | none | \( \alpha \) controls shrinkage strength |

---

## Reason to use Ridge Regression
> Ridge regression sacrifices a tiny bit of fit accuracy on the training data in exchange for better generalization and more stable coefficients — especially useful when predictors overlap or data are noisy.



In [ ]:

for alpha in [0.0, 0.1, 1.0, 10.0]:
    ridge = Ridge(alpha=alpha, fit_intercept=True)
    ridge.fit(x.reshape(-1,1), y)
    print(f'Ridge(alpha={alpha:.1f}) -> b = {ridge.intercept_:.3f}, w = {ridge.coef_[0]:.3f}, R^2 = {ridge.score(x.reshape(-1,1), y):.3f}')

# Plot different alphas
plt.figure(figsize=(6,4))
plt.scatter(x, y, alpha=0.6, label='data')
for alpha in [0.1, 1, 10.0, 100.0, 0.0]:
    ridge = Ridge(alpha=alpha, fit_intercept=True)
    ridge.fit(x.reshape(-1,1), y)
    plt.plot(x_line, ridge.predict(x_line.reshape(-1,1)), linewidth=2, label=f'alpha={alpha}')
plt.xlabel('x'); plt.ylabel('y'); plt.title('Ridge Regression: Effect of L2 Penalty')
plt.legend()
plt.show()



## 7) Residual analysis (quick check)

Residuals should look roughly like zero-mean noise if the linear model is well-specified.


In [ ]:

resid = y - y_pred_skl
plt.figure(figsize=(6,4))
plt.scatter(x, resid, alpha=0.7)
plt.axhline(0, linestyle='--')
plt.xlabel('x'); plt.ylabel('Residual')
plt.title('Residuals vs x (sklearn fit)')
plt.show()

plt.figure(figsize=(6,4))
plt.hist(resid, bins=20)
plt.xlabel('Residual'); plt.ylabel('Count')
plt.title('Residual distribution')
plt.show()



### Summary
- **Closed-form** and **gradient descent** arrive at the same solution (given convergence).
- **R²** tells you how much variance your model explains.
- **Ridge** (L2) can stabilize estimates when noise or *multicollinearity* is present.

**[Optional Next Steps] Play with the simmulated data and see how the models respond:** try adding outliers, heteroscedastic noise, or a nonlinear relationship and see how the model behaves.



## 8) Multivariate Linear Regression (Multiple Predictors)

We'll extend to two predictors, `x1` and `x2`, and fit a plane:
\[ y \approx b + w_1 x_1 + w_2 x_2 + \epsilon \]

We'll show:
1. Closed-form solution in matrix form.
2. `scikit-learn` fit and metrics.
3. **Interactive 3D plot** using Plotly — drag to rotate axes.


In [ ]:

# If running in Colab, ensure plotly is available
try:
    import plotly
except ImportError:
    !pip -q install plotly

import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(7)

# Generate multivariate synthetic data
n = 200
x1 = rng.uniform(-3, 3, size=n)
x2 = rng.uniform(-3, 3, size=n)
noise = rng.normal(0, 1.0, size=n)
# True relationship: y = 2 + 1.5*x1 - 0.8*x2 + noise
y2 = 2.0 + 1.5*x1 - 0.8*x2 + noise

# write data to file
df = pd.DataFrame({'x1': x1,'x2': x2, 'y2': y2})
df.to_csv('multireg_data.csv', index=False)   # export the simulated data to csv file
print('Saved data to multireg_data.csv')


# Design matrix with bias
X2 = np.c_[np.ones(n), x1, x2]
beta_hat2 = np.linalg.inv(X2.T @ X2) @ (X2.T @ y2.reshape(-1,1))
b2, w1, w2 = beta_hat2.ravel()
print(f'Closed-form (two predictors): b = {b2:.3f}, w1 = {w1:.3f}, w2 = {w2:.3f}')

# scikit-learn comparison
lm2 = LinearRegression()
lm2.fit(np.c_[x1, x2], y2)
print(f'sklearn -> b = {lm2.intercept_:.3f}, w1 = {lm2.coef_[0]:.3f}, w2 = {lm2.coef_[1]:.3f}')
print(f'R^2 (sklearn) = {lm2.score(np.c_[x1, x2], y2):.3f}')

# Create a meshgrid for the regression plane (using sklearn parameters)
g = np.linspace(-3, 3, 40)
gx1, gx2 = np.meshgrid(g, g)
gy = lm2.intercept_ + lm2.coef_[0]*gx1 + lm2.coef_[1]*gx2

# 3D interactive plot (rotatable axes)
fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=x1, y=x2, z=y2,
    mode='markers',
    marker=dict(size=3),
    name='Data'
))

fig.add_trace(go.Surface(
    x=gx1, y=gx2, z=gy,
    opacity=0.6,
    showscale=False,
    name='Regression plane'
))

fig.update_layout(
    title='Multivariate Linear Regression (Interactive 3D)',
    scene=dict(
        xaxis_title='x1',
        yaxis_title='x2',
        zaxis_title='y',
        aspectmode='cube'
    ),
    width=700,
    height=700
)

fig.show()



### Notes
- The Plotly figure above is **fully rotatable** — click and drag to change the view; scroll or trackpad to zoom.
- The fitted plane uses the `scikit-learn` parameters; closed-form and sklearn should closely match.
- Try changing the true coefficients or adding correlation between `x1` and `x2` to see effects on estimates.


In [ ]:
#@title Setup & synthetic 2D data
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

# latent factor -> creates correlation between x and y
n = 200
z = rng.normal(0, 1, n)
eps_x = rng.normal(0, 0.6, n)
eps_y = rng.normal(0, 0.8, n)

# Make x,y with a linear relationship but different noise scales
x = 2.0*z + eps_x
y = 1.2*x + 3.0 + eps_y  # true slope ~1.2, intercept ~3.0

X = np.column_stack([x, y])
x_mean, y_mean = x.mean(), y.mean()
Xc = X - X.mean(axis=0)  # centered for PCA


## Principal Component Analysis (PCA)

**Principal Component Analysis (PCA)** is a mathematical technique that simplifies complex, high-dimensional datasets by finding new coordinate axes — called principal components (PCs) — that capture the maximum variance in the data.

---

### Step-by-step intuition

1. **Start with standardized features**  
   PCA begins by centering and scaling the data so that all variables have mean = 0 and variance = 1.  
   (Without this, variables measured in different units — e.g., mRNA counts vs. metabolite concentrations — would dominate unfairly.)

2. **Find the directions of greatest variance**  
   PCA looks for a new axis (a linear combination of the original features) that captures as much variation in the data as possible. "Weights" here are analagous to the "slopes/betas" in regression.

   Mathematically, it solves for weight vectors \( $\mathbf{w}_1$, $\mathbf{w}_2$, … \) that maximize:

   $$
   \text{Var}(X\mathbf{w}) \quad \text{subject to } \|\mathbf{w}\| = 1
   $$

3. **Orthogonal components**  
   After finding the first principal component (PC1), PCA finds another axis (PC2) that:
   - Captures the next-largest amount of variance  
   - Is **orthogonal** (uncorrelated) to PC1

4. **Project the data**  
   Each sample’s coordinates along PC1, PC2, … are computed as:

   $$
   Z = X \mathbf{W}
   $$

   where  
   • \($ X $\) = standardized data matrix  
   • \( $\mathbf{W} $\) = matrix of eigenvectors (principal axes)  
   • \($ Z $\) = transformed data in the new PCA space

---

### Connection to linear algebra

- PCA is computed using **eigen-decomposition** or **Singular Value Decomposition (SVD)**.  
- The eigenvectors of the covariance matrix \($ X^\top X $\) define the directions of principal components.  
- The eigenvalues tell how much variance each component explains.

---

### What PCA gives you

| Output | Meaning |
|---------|----------|
| **PC scores** | The coordinates of each sample in the new PCA space (used for plotting) |
| **Loadings** | How strongly each original feature contributes to each PC |
| **Explained variance ratio** | Fraction of total variance captured by each component |

---

### What's it doing?

> PCA rotates the data to a new set of axes that best describe how the samples vary. The first few principal components usually capture most of the biological or experimental structure, letting us visualize patterns, detect outliers, and reduce noise.


In [ ]:
#@title Fit regression & PCA; compute lines
from sklearn.linear_model import LinearRegression
from numpy.linalg import svd

# Linear regression (y ~ x)
lm = LinearRegression()
lm.fit(x.reshape(-1,1), y)
b_lr = lm.intercept_
w_lr = lm.coef_[0]

# PCA via SVD on centered data (Xc)
# Xc = U S V^T, rows are samples, columns are [x, y]
U, S, Vt = svd(Xc, full_matrices=False)
pc1 = Vt[0]             # first principal component direction in (x,y) coords
vx, vy = pc1            # direction vector
# slope of PC1 line in x-y plane (avoid divide by zero)
slope_pc = vy / vx if np.abs(vx) > 1e-12 else np.inf
# PC1 line passes through the mean (x_mean, y_mean)
b_pc = y_mean - slope_pc * x_mean if np.isfinite(slope_pc) else None

# Prepare a plotting range
xs = np.linspace(x.min()-0.5, x.max()+0.5, 200)
y_lr = b_lr + w_lr*xs
y_pc = (b_pc + slope_pc*xs) if np.isfinite(slope_pc) else np.full_like(xs, np.nan)

print(f"Regression:   y = {w_lr:.3f} * x + {b_lr:.3f}")
if np.isfinite(slope_pc):
    print(f"PCA (PC1):    y = {slope_pc:.3f} * x + {b_pc:.3f}  (line through data mean)")
else:
    print("PCA (PC1):    vertical line through x = x_mean")


In [ ]:
#@title Plot data with regression line & PC1 line, plus residual segments
plt.figure(figsize=(7,6))
plt.scatter(x, y, s=18, alpha=0.7, label="data")

# Regression line
plt.plot(xs, y_lr, lw=2, label="Linear regression (minimize vertical error)")

# PC1 line (variance-maximizing direction)
if np.isfinite(slope_pc):
    plt.plot(xs, y_pc, lw=2, linestyle="--", label="PCA PC1 (variance direction)")
else:
    plt.axvline(x_mean, lw=2, linestyle="--", label="PCA PC1 (vertical line)")

# Show residual segments for a small random subset to avoid clutter
idx = rng.choice(len(x), size=15, replace=False)

# Vertical residuals to regression line (blue, dotted)
for i in idx:
    y_hat = b_lr + w_lr*x[i]
    plt.plot([x[i], x[i]], [y[i], y_hat], color="#2563eb", ls=":", lw=1.8)

# Orthogonal residuals to PC1 line (red, dotted)
if np.isfinite(slope_pc):
    # Unit direction vector along PC1 and its normal
    v = np.array([vx, vy])
    v = v / (np.linalg.norm(v) + 1e-12)
    nvec = np.array([-v[1], v[0]])  # normal to PC1
    mu = np.array([x_mean, y_mean])

    for i in idx:
        p = np.array([x[i], y[i]])
        # projection point on PC1:
        t = np.dot(p - mu, v)
        proj = mu + t * v
        plt.plot([p[0], proj[0]], [p[1], proj[1]], color="#dc2626", ls=":", lw=1.8)

plt.xlabel("x"); plt.ylabel("y")
plt.title("Regression vs PCA: two different projections")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#@title Compare "errors": vertical vs orthogonal-to-PC1
# Vertical residuals to regression line
y_hat_lr = b_lr + w_lr*x
mse_vertical = np.mean((y - y_hat_lr)**2)

# Orthogonal distances to PC1 line
if np.isfinite(slope_pc):
    v = np.array([vx, vy])
    v = v / (np.linalg.norm(v) + 1e-12)
    mu = np.array([x_mean, y_mean])
    P = np.column_stack([x, y])
    # vector from mean to each point
    P0 = P - mu
    # parallel component length along v
    t = P0 @ v
    proj = mu + np.outer(t, v)
    orth_dists = np.linalg.norm(P - proj, axis=1)
    mean_orth = orth_dists.mean()
else:
    # vertical PC1: orthogonal distance is horizontal distance to x_mean
    mean_orth = np.mean(np.abs(x - x_mean))

print(f"MSE (vertical, regression): {mse_vertical:.3f}")
print(f"Mean orthogonal distance to PC1: {mean_orth:.3f}")


In [ ]:
#@title Optional: Principal Component Regression (using PC1 as 1-D summary of X)
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Build a 2D feature matrix for PCR (Xc already centered)
# PC1 scores: U[:,0]*S[0] equals Xc @ pc1   (up to sign)
pc1_scores = Xc @ pc1  # 1-D representation of (x,y)

# Compare y ~ x   vs   y ~ PC1
X_train, X_test, y_train, y_test = train_test_split(
    x.reshape(-1,1), y, test_size=0.3, random_state=42
)
lm_x = LinearRegression().fit(X_train, y_train)
pred_x = lm_x.predict(X_test)

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    pc1_scores.reshape(-1,1), y, test_size=0.3, random_state=42
)
lm_p = LinearRegression().fit(X_train_p, y_train_p)
pred_p = lm_p.predict(X_test_p)

print("y ~ x     -> R^2 = %.3f, RMSE = %.3f" % (r2_score(y_test, pred_x), np.sqrt(mean_squared_error(y_test, pred_x))))
print("y ~ PC1   -> R^2 = %.3f, RMSE = %.3f" % (r2_score(y_test_p, pred_p), np.sqrt(mean_squared_error(y_test_p, pred_p))))


#Multiple PCs example 5 varables to predict enzyme activity

* GeneA_expression     
* GeneB_expression     
* Metabolite1_concentration    
* StressMarker_level
* SignalProtein_level




In [ ]:
#@title Simulated biological dataset (5 predictors + target)
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(123)
n = 400

# latent biological factors
metabolic_state = rng.normal(0, 1.0, n)
stress_response = rng.normal(0, 1.0, n)
signaling_state = rng.normal(0, 1.0, n)

# generate correlated biological features
GeneA_expression     = 2.0*metabolic_state + 0.5*stress_response + rng.normal(0, 0.5, n)
GeneB_expression     = 1.5*metabolic_state - 0.3*stress_response + rng.normal(0, 0.5, n)
Metabolite1_conc     = 0.2*metabolic_state + 1.8*stress_response + rng.normal(0, 0.6, n)
StressMarker_level   = -0.5*metabolic_state + 0.4*signaling_state + rng.normal(0, 0.7, n)
SignalProtein_level  = 0.1*stress_response + 1.0*signaling_state + rng.normal(0, 0.7, n)

# stack into feature matrix
X = np.column_stack([GeneA_expression, GeneB_expression,
                     Metabolite1_conc, StressMarker_level,
                     SignalProtein_level])

# true biological relationship -> enzyme activity
true_w = np.array([1.2, 0.0, 0.8, 0.4, 0.0])
enzyme_activity = X @ true_w + 3.0 + rng.normal(0, 0.8, n)

# name columns
cols = ["GeneA_expression","GeneB_expression",
        "Metabolite1_conc","StressMarker_level",
        "SignalProtein_level","Enzyme_activity"]

# build DataFrame and save
df = pd.DataFrame(np.column_stack([X, enzyme_activity]), columns=cols)
df.to_csv("simulated_biology_dataset.csv", index=False)
print(f"✅ Saved simulated_biology_dataset.csv with shape {df.shape}")
df.head()


In [ ]:
#@title Compute PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Separate features (X) and target (y)
feature_cols = ["GeneA_expression","GeneB_expression",
                "Metabolite1_conc","StressMarker_level",
                "SignalProtein_level"]
target_col = "Enzyme_activity"

X = df[feature_cols].values
y = df[target_col].values

# Standardize features
Xz = StandardScaler().fit_transform(X)

# Perform PCA
pca = PCA(n_components=2, random_state=42)
PC = pca.fit_transform(Xz)

# Extract PC1 and PC2
pc1, pc2 = PC[:, 0], PC[:, 1]

print("Explained variance by PC1 & PC2:",
      np.round(pca.explained_variance_ratio_, 3))


In [ ]:
#@title 2D PC1 vs PC2 scatter (colored by y)
plt.figure(figsize=(6.5, 5.5))
sc = plt.scatter(pc1, pc2, c=y, cmap="viridis", s=16, alpha=0.9)
plt.colorbar(sc, label="Enzyme Activity")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PC1 vs PC2 (color = Enzyme Activity)")
plt.tight_layout()
plt.show()


In [ ]:
#@title Rotatable 3D plot: PC1 vs PC2 vs y (aspect = cube)
# If running in Colab, ensure plotly is available
try:
    import plotly
except ImportError:
    !pip -q install plotly

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=pc1, y=pc2, z=y,
    mode='markers',
    marker=dict(size=3, color=y, colorscale='Viridis', showscale=True, opacity=0.9),
    name='samples'
))

fig.update_layout(
    title='Rotatable PCA space: PC1 vs PC2 vs Enzyme Activity',
    scene=dict(
        xaxis_title='PC1',
        yaxis_title='PC2',
        zaxis_title='Enzyme Activity',
        aspectmode='cube'   # 🔹 makes the box a cube
    ),
    width=750, height=750
)

fig.show()


In [ ]:
#@title Compare y ~ (PC1, PC2) vs y ~ X (original 5 features)
from sklearn.metrics import r2_score, mean_squared_error

# PCR with first 2 PCs
X_pcr = np.column_stack([pc1, pc2])
lm_pcr = LinearRegression().fit(X_pcr, y)
yhat_pcr = lm_pcr.predict(X_pcr)

# Full model with original features
lm_full = LinearRegression().fit(X, y)
yhat_full = lm_full.predict(X)

def report(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name:16s}  RMSE = {rmse:.3f}   R² = {r2:.3f}")

report("PCR (PC1+PC2)", y, yhat_pcr)
report("Full (5 feats)", y, yhat_full)
print("True weights:", np.round(true_w, 3))
print("Full-model est. weights:", np.round(lm_full.coef_, 3), "  intercept:", round(lm_full.intercept_, 3))


In [ ]:
#@title PCA fit + loadings table (biological features)
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Separate features (X) and target (y)
feature_cols = ["GeneA_expression","GeneB_expression",
                "Metabolite1_conc","StressMarker_level",
                "SignalProtein_level"]
target_col = "Enzyme_activity"

X = df[feature_cols].values
y = df[target_col].values

# Standardize features before PCA
scaler = StandardScaler()
Xz = scaler.fit_transform(X)

# PCA -> 2 components
pca = PCA(n_components=2, random_state=42)
scores = pca.fit_transform(Xz)     # sample projections (PC scores)
loadings = pca.components_.T       # shape: (n_features, n_components)

explained = pca.explained_variance_ratio_
print("Explained variance ratio:", np.round(explained, 3))

# Loadings table
loadings_df = pd.DataFrame(loadings, index=feature_cols, columns=["PC1_loading","PC2_loading"])
loadings_df["loading_norm"] = np.sqrt(loadings_df["PC1_loading"]**2 + loadings_df["PC2_loading"]**2)
loadings_df.sort_values("loading_norm", ascending=False)


In [ ]:
#@title PCA biplot (PC1 vs PC2) with variable arrows
import matplotlib.pyplot as plt

pc1, pc2 = scores[:,0], scores[:,1]

plt.figure(figsize=(7,6))

# (Optional) color samples by target to connect unsupervised PCs to y
sc = plt.scatter(pc1, pc2, c=y, s=20, cmap="viridis", alpha=0.85)
cbar = plt.colorbar(sc)
cbar.set_label(target_col)

# Arrow scaling so vectors are visible relative to the score spread
# Heuristic: scale arrows to ~40% of the data range
scale = 0.4 * max(pc1.max()-pc1.min(), pc2.max()-pc2.min())

# Draw loading vectors (arrows) from origin
for i, name in enumerate(feature_cols):
    vx = loadings[i,0] * scale
    vy = loadings[i,1] * scale
    plt.arrow(0, 0, vx, vy, color="crimson", width=0.0, head_width=0.07*scale,
              length_includes_head=True, alpha=0.9)
    # Label slightly beyond the arrow tip
    offset = 0.06*scale
    plt.text(vx + offset, vy + offset, name, color="crimson", fontsize=11)

plt.axhline(0, color="#999", lw=1)
plt.axvline(0, color="#999", lw=1)
plt.xlabel(f"PC1 ({explained[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({explained[1]*100:.1f}% var)")
plt.title("PCA Biplot: samples in PC space with variable loadings")
plt.tight_layout()
plt.show()


In [ ]:
#@title Variable contributions to PC1 and PC2 (absolute loadings)
abs_load = loadings_df[["PC1_loading","PC2_loading"]].abs().copy()
abs_load.sort_values("PC1_loading", ascending=False, inplace=True)

plt.figure(figsize=(7,4))
plt.bar(abs_load.index, abs_load["PC1_loading"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("|loading|")
plt.title("Absolute loadings on PC1 (variable contribution)")
plt.tight_layout()
plt.show()

abs_load2 = loadings_df[["PC1_loading","PC2_loading"]].abs().copy()
abs_load2.sort_values("PC2_loading", ascending=False, inplace=True)

plt.figure(figsize=(7,4))
plt.bar(abs_load2.index, abs_load2["PC2_loading"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("|loading|")
plt.title("Absolute loadings on PC2 (variable contribution)")
plt.tight_layout()
plt.show()


In [ ]:
#@title PCA to 3 components
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

feature_cols = ["GeneA_expression","GeneB_expression",
                "Metabolite1_conc","StressMarker_level",
                "SignalProtein_level"]
target_col = "Enzyme_activity"

X = df[feature_cols].values
y = df[target_col].values

scaler = StandardScaler()
Xz = scaler.fit_transform(X)

pca3 = PCA(n_components=3, random_state=42)
scores3 = pca3.fit_transform(Xz)
pc1, pc2, pc3 = scores3[:,0], scores3[:,1], scores3[:,2]

print("Explained variance (PC1..PC3):", np.round(pca3.explained_variance_ratio_, 3))


In [ ]:
#@title 3D PCA scatter (PC1, PC2, PC3) colored by Enzyme_activity
try:
    import plotly
except ImportError:
    !pip -q install plotly

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=pc1, y=pc2, z=pc3,
    mode='markers',
    marker=dict(size=3, color=y, colorscale='Viridis', showscale=True, opacity=0.9,
                colorbar=dict(title=target_col)),
    name='samples'
))

fig.update_layout(
    title='PCA (3D): PC1 vs PC2 vs PC3 (color = Enzyme_activity)',
    scene=dict(
        xaxis_title=f"PC1 ({pca3.explained_variance_ratio_[0]*100:.1f}% var)",
        yaxis_title=f"PC2 ({pca3.explained_variance_ratio_[1]*100:.1f}% var)",
        zaxis_title=f"PC3 ({pca3.explained_variance_ratio_[2]*100:.1f}% var)",
        aspectmode='cube'   # keep it cubic
    ),
    width=750, height=750
)

fig.show()


In [ ]:
#@title 3D loading arrows (top contributors)
loadings3 = pca3.components_.T   # shape (n_features, 3)

# scale arrows to ~40% of the score spread
rng_span = max(pc1.max()-pc1.min(), pc2.max()-pc2.min(), pc3.max()-pc3.min())
scale = 0.4 * rng_span

# pick top variables by vector length in PC1..PC3
strength = np.linalg.norm(loadings3, axis=1)
order = np.argsort(strength)[::-1]  # descending
top = order[:5]  # show all five here since you only have 5 vars

for i in top:
    vx, vy, vz = loadings3[i] * scale
    fig.add_trace(go.Scatter3d(
        x=[0, vx], y=[0, vy], z=[0, vz],
        mode='lines+text',
        line=dict(color='crimson', width=6),
        text=[None, feature_cols[i]],
        textposition='top center',
        name=feature_cols[i],
        showlegend=False
    ))

fig.update_layout(title=fig.layout.title.text + " + Loadings")
fig.show()
